In [1]:
# root file producer for the QC-TS
import os
import ROOT as root
import numpy as np
from array import array
import glob
import re
import pandas as pd
from openpyxl import Workbook
import pytz

Welcome to JupyROOT 6.30/04


In [ ]:
file = root.TFile("/Users/icosivi/Desktop/PRE-SERIE/HPK/PRE-SERIE_HPK_Vendor_QC-TS_CV.root", "RECREATE")
tree = root.TTree("Tree","Tree")

event = array('i', [0])
wafer = array('i', [0])
sensor = array('i', [0])
side = array('i', [0])

V = root.std.vector("float")()
CPAD = root.std.vector("float")()

tree.Branch("wafer", wafer, 'wafer/I')
tree.Branch("event", event, 'event/I')
tree.Branch("sensor", sensor,'sensor/I')
tree.Branch("side", side,'side/I')


V.reserve(1000)
CPAD.reserve(1000)
tree.Branch("V", "std::vector<float>", V)
tree.Branch("CPAD", "std::vector<float>", CPAD)

csv_files = glob.glob("/Users/icosivi/Desktop/PRE-SERIE/HPK/on-wafer_data/*.xlsx")

nevent = 0

for csv in csv_files:
  #print(csv)
  df = pd.read_excel(csv, sheet_name='CV', header=None)
  nome_senza_ext = os.path.splitext(csv)[0]
  wafer[0] = int(nome_senza_ext.split('-')[-1])

  for i in range(8):
      V.clear()
      CPAD.clear()
      
      event[0] = nevent
      sensor[0] = int(i+1)
      side[0] = 0
      
      C_list = df.iloc[2:,i+1].dropna().tolist()
      if C_list:
        subset = df.iloc[2:, [0, i+1]].dropna() 
        for _, rr in subset.iterrows():  
          CPAD.push_back( float(rr.iloc[1]) )
          V.push_back( float(rr.iloc[0]) )
        
      tree.Fill()
      nevent += 1
      
      
      #V_list = df[3:,0].dropna().tolist()
      #for v in V_list:
      #  V.push_back( float(v) )

tree.Write()
file.Write()
file.Close()

In [2]:
# producer of the xls to register components on the database for the QC-TS
xl_filename="HPK_QC-TS_PRE-SERIES"
wb = Workbook()
wws = wb.active
wws["A1"] = "SerialNumber"
wws["B1"] = "Vendor" 
wws["C1"] = "Batch" 
wws["D1"] = "Wafer" 
wws["E1"] = "Geometry"
wws["F1"] = "Row" 
wws["G1"] = "Column"
wws["H1"] = "Sensor Number"

batch = int()

file_qa = root.TFile.Open("/Users/icosivi/Desktop/PRE-SERIE/HPK/PRE-SERIE_HPK_Vendor_QC-TS_CV.root")
tree_qa = file_qa.Get("Tree")

for j,event in enumerate(tree_qa):
    
    if event.wafer<48:
        batch = 1
    else:
        batch = 2

    wws["A%i" %(j+2)] = 'PRE_HPK_QC-TS_W'+str(event.wafer)+'_S'+str(event.sensor)
        
    wws["B%i" %(j+2)] = 'HPK'
    wws["C%i" %(j+2)] = batch
    wws["D%i" %(j+2)] = event.wafer
    wws["E%i" %(j+2)] = 'QC-TS'
    wws["F%i" %(j+2)] = None
    wws["G%i" %(j+2)] = None
    wws["H%i" %(j+2)] = event.sensor

save_path = '/Users/icosivi/Desktop/PRE-SERIE/HPK/'
wb.save(save_path+xl_filename+".xlsx")